# Auditoria-FINAL-POPCORRIGIDA-X-CASOS

In [6]:
import pandas as pd
import numpy as np
import re
import json
import unicodedata
from pathlib import Path

# ============================================================
# CONFIGURAÇÕES
# ============================================================

ARQ_POP = "PopulaçãoCorrigida_interpolada_CAGR_FINAL.xlsx"

ARQ_CASOS_SP = "Casos-TB-SP-NOVO.csv"
ARQ_CASOS_RJ = "Casos-TB-RJ-NOVO.csv"

ARQ_GEOJSON_SP = "SP_geojs-35-mun.json"
ARQ_GEOJSON_RJ = "RJ_geojs-33-mun.json"

OUTFILE = "Auditoria_populacao_e_pequenos_numeros_TB_SP_RJ_CORRIGIDA.xlsx"

ANOS_POP = list(range(2000, 2025))
ANOS_CASOS = list(range(2001, 2025))

INTERVALOS_CAGR = [
    (2000, 2010),
    (2010, 2022),
    (2022, 2024),
]

MUNICIPIOS_ESPERADOS = {
    "RJ": 92,
    "SP": 645,
}

# Limiares de auditoria populacional
LIMIAR_SALTO_ANUAL_10 = 10
LIMIAR_SALTO_ANUAL_20 = 20
LIMIAR_CAGR_EXTREMO = 5

# Limiares da lei dos pequenos números
POP_MUITO_PEQUENA = 10_000
POP_PEQUENA = 20_000
POP_MODERADA = 50_000

# Incidência por 100 mil habitantes
LIMIAR_INCIDENCIA_ALTA = 100
LIMIAR_INCIDENCIA_MUITO_ALTA = 300


# ============================================================
# FUNÇÕES GERAIS
# ============================================================

def remover_acentos(txt):
    txt = str(txt)
    return "".join(
        c for c in unicodedata.normalize("NFKD", txt)
        if not unicodedata.combining(c)
    )


def norm_txt(x):
    x = remover_acentos(str(x)).strip().lower()
    x = re.sub(r"\s+", "_", x)
    return x


def limpar_numero(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    x = str(x).strip()

    if x == "":
        return np.nan

    if x in ["-", "–", "—"]:
        return 0.0

    x = x.replace("\xa0", " ")
    x = x.replace(" ", "")
    x = x.replace(".", "")
    x = x.replace(",", ".")

    try:
        return float(x)
    except ValueError:
        return np.nan


def limpar_casos(x):
    """
    Nos arquivos do DATASUS/TABNET, '-' geralmente indica zero.
    """
    if pd.isna(x):
        return 0.0

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    x = str(x).strip()

    if x in ["", "-", "–", "—"]:
        return 0.0

    x = x.replace("\xa0", " ")
    x = x.replace(" ", "")
    x = x.replace(".", "")
    x = x.replace(",", ".")

    try:
        return float(x)
    except ValueError:
        return 0.0


def limpar_codigo_municipio(x):
    """
    Padroniza código municipal para 6 dígitos.
    Exemplo:
    3300100 -> 330010
    330010  -> 330010
    '330010 ANGRA DOS REIS' -> 330010
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.replace(".0", "")

    m = re.search(r"(\d+)", x)

    if not m:
        return np.nan

    cod = m.group(1)

    if len(cod) >= 6:
        return cod[:6].zfill(6)

    return cod.zfill(6)


def detectar_estado_por_codigo(cod):
    cod = str(cod)

    if cod.startswith("33"):
        return "RJ"

    if cod.startswith("35"):
        return "SP"

    return np.nan


def achar_coluna(df, padroes, excluir=None):
    if excluir is None:
        excluir = []

    excluir = set([c for c in excluir if c is not None])

    for col in df.columns:
        if col in excluir:
            continue

        col_norm = norm_txt(col)

        for padrao in padroes:
            if re.search(padrao, col_norm):
                return col

    return None


def achar_coluna_codigo(df):
    col = achar_coluna(df, [
        r"^cod_mun$",
        r"^codmun$",
        r"^cod_municipio$",
        r"^codigo_municipio$",
        r"^codigo_ibge$",
        r"^cod_ibge$",
        r"^geocodigo$",
        r"^id_municipio$",
        r"^ibge$",
        r"^codigo$",
        r"^cod$",
    ])

    if col is not None:
        return col

    # Tentativa por conteúdo
    for c in df.columns:
        s = (
            df[c]
            .dropna()
            .astype(str)
            .str.replace(r"\.0$", "", regex=True)
            .str.extract(r"(\d+)", expand=False)
            .dropna()
        )

        if len(s) == 0:
            continue

        prop_codigos = s.str.match(r"^\d{6,7}$").mean()

        if prop_codigos > 0.8:
            return c

    return None


def achar_coluna_municipio(df, excluir=None):
    if excluir is None:
        excluir = []

    col = achar_coluna(df, [
        r"^municipio$",
        r"^municipios$",
        r"^nome_municipio$",
        r"^nome_do_municipio$",
        r"^nome_mun$",
        r"^nome$",
        r"municipio_de_notificacao",
        r"municipio_notificacao",
        r"mun_not",
    ], excluir=excluir)

    if col is not None:
        return col

    # Tentativa por conteúdo textual
    for c in df.columns:
        if c in excluir:
            continue

        serie = df[c].dropna().astype(str)

        if len(serie) == 0:
            continue

        prop_numerica = serie.str.match(r"^\d+(\.0)?$").mean()

        if prop_numerica > 0.5:
            continue

        prop_texto = serie.str.contains(r"[A-Za-zÁ-Úá-ú]", regex=True).mean()

        if prop_texto > 0.8:
            return c

    return None


def achar_coluna_ano(df, ano):
    for col in df.columns:
        if str(col).strip() == str(ano):
            return col

    for col in df.columns:
        if norm_txt(col) == str(ano):
            return col

    return None


def safe_sheet_name(nome):
    nome = str(nome)
    nome = re.sub(r"[\[\]\:\*\?\/\\]", "_", nome)
    return nome[:31]


def mostrar(df, n=20):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n))


def achar_arquivo(nome_arquivo, padrao_busca=None):
    path = Path(nome_arquivo)

    if path.exists():
        return path

    if padrao_busca is not None:
        encontrados = list(Path(".").glob(padrao_busca))
        if len(encontrados) > 0:
            return encontrados[0]

    raise FileNotFoundError(f"Arquivo não encontrado: {nome_arquivo}")


# ============================================================
# LEITURA DA POPULAÇÃO
# ============================================================

def aba_parece_populacao(df):
    col_cod = achar_coluna_codigo(df)
    col_mun = achar_coluna_municipio(df, excluir=[col_cod])

    n_anos = 0
    for ano in ANOS_POP:
        if achar_coluna_ano(df, ano) is not None:
            n_anos += 1

    return col_cod is not None and col_mun is not None and n_anos >= 20


def padronizar_aba_populacao(df, nome_aba):
    df = df.copy()

    col_cod = achar_coluna_codigo(df)
    col_mun = achar_coluna_municipio(df, excluir=[col_cod])

    col_uf = achar_coluna(df, [
        r"^uf$",
        r"^estado$",
        r"^sigla$",
    ], excluir=[col_cod, col_mun])

    if col_cod is None:
        raise ValueError(f"Não encontrei coluna de código municipal na aba {nome_aba}")

    if col_mun is None:
        raise ValueError(f"Não encontrei coluna de município na aba {nome_aba}")

    print(f"  Coluna código detectada: {col_cod}")
    print(f"  Coluna município detectada: {col_mun}")

    df = df.rename(columns={
        col_cod: "cod_mun_original",
        col_mun: "municipio",
    })

    if col_uf is not None:
        df = df.rename(columns={col_uf: "estado_original"})
    else:
        df["estado_original"] = np.nan

    renomear_anos = {}

    for ano in ANOS_POP:
        col_ano = achar_coluna_ano(df, ano)
        if col_ano is not None:
            renomear_anos[col_ano] = ano

    df = df.rename(columns=renomear_anos)

    for ano in ANOS_POP:
        if ano not in df.columns:
            df[ano] = np.nan

    df["cod_mun6"] = df["cod_mun_original"].apply(limpar_codigo_municipio)
    df["estado"] = df["cod_mun6"].apply(detectar_estado_por_codigo)

    df["municipio"] = (
        df["municipio"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    for ano in ANOS_POP:
        df[ano] = df[ano].apply(limpar_numero)

    df = df[["estado", "cod_mun6", "municipio"] + ANOS_POP]

    df = df[df["estado"].isin(["RJ", "SP"])].copy()

    return df


def carregar_populacao(path):
    path = achar_arquivo(path, "*Popula*Corrigida*CAGR*FINAL*.xlsx")

    print(f"Arquivo de população usado: {path}")

    abas = pd.read_excel(path, sheet_name=None)

    partes = []
    abas_usadas = []
    abas_ignoradas = []

    for nome_aba, df in abas.items():
        print(f"\nAvaliando aba: {nome_aba}")

        if aba_parece_populacao(df):
            print(f"Aba usada: {nome_aba}")
            partes.append(padronizar_aba_populacao(df, nome_aba))
            abas_usadas.append(nome_aba)
        else:
            print(f"Aba ignorada: {nome_aba}")
            abas_ignoradas.append(nome_aba)

    if len(partes) == 0:
        raise ValueError("Nenhuma aba populacional foi identificada.")

    pop_wide = pd.concat(partes, ignore_index=True)

    pop_wide = (
        pop_wide
        .drop_duplicates(["estado", "cod_mun6"], keep="first")
        .sort_values(["estado", "cod_mun6"])
        .reset_index(drop=True)
    )

    return pop_wide, abas_usadas, abas_ignoradas


def populacao_wide_para_long(pop_wide):
    pop_long = pop_wide.melt(
        id_vars=["estado", "cod_mun6", "municipio"],
        value_vars=ANOS_POP,
        var_name="ano",
        value_name="populacao"
    )

    pop_long["ano"] = pop_long["ano"].astype(int)
    pop_long["populacao"] = pd.to_numeric(pop_long["populacao"], errors="coerce")

    pop_long = (
        pop_long
        .sort_values(["estado", "cod_mun6", "ano"])
        .reset_index(drop=True)
    )

    pop_long["populacao_anterior"] = (
        pop_long
        .groupby(["estado", "cod_mun6"])["populacao"]
        .shift(1)
    )

    pop_long["var_pop_abs"] = (
        pop_long["populacao"] - pop_long["populacao_anterior"]
    )

    pop_long["var_pop_pct"] = np.where(
        pop_long["populacao_anterior"] > 0,
        100 * pop_long["var_pop_abs"] / pop_long["populacao_anterior"],
        np.nan
    )

    return pop_long


# ============================================================
# LEITURA DOS CASOS - CORRIGIDA PARA FORMATO WIDE
# ============================================================

def ler_csv_auto(path):
    path = achar_arquivo(path)

    tentativas = [
        {"sep": ";", "encoding": "latin1"},
        {"sep": ";", "encoding": "utf-8"},
        {"sep": ",", "encoding": "latin1"},
        {"sep": ",", "encoding": "utf-8"},
    ]

    ultimo_erro = None

    for kwargs in tentativas:
        try:
            df = pd.read_csv(path, **kwargs)

            if df.shape[1] > 1:
                print(f"\nArquivo lido: {path.name}")
                print(f"Separador: {kwargs['sep']} | Encoding: {kwargs['encoding']}")
                print(f"Nº linhas: {df.shape[0]} | Nº colunas: {df.shape[1]}")
                print(f"Primeiras colunas: {list(df.columns)[:8]}")
                return df

        except Exception as e:
            ultimo_erro = e

    raise ValueError(f"Não consegui ler o CSV: {path}. Último erro: {ultimo_erro}")


def detectar_coluna_municipio_casos_wide(df):
    """
    Detecta coluna do tipo:
    'Município de notificação'
    com valores como:
    '350010 ADAMANTINA'
    """

    # 1. Detecta pelo nome
    for col in df.columns:
        col_norm = norm_txt(col)

        if "municipio" in col_norm or "munic" in col_norm:
            return col

    # 2. Detecta pelo conteúdo
    for col in df.columns:
        serie = df[col].dropna().astype(str)

        if len(serie) == 0:
            continue

        prop_codigo_nome = serie.str.match(r"^\s*\d{6,7}\s+.+").mean()

        if prop_codigo_nome > 0.6:
            return col

    raise ValueError("Não consegui identificar a coluna de município no arquivo de casos.")


def detectar_colunas_anos_casos_wide(df):
    """
    Detecta colunas 2001, 2002, ..., 2024.
    Ignora Total e colunas descritivas.
    """

    colunas_anos = []

    for col in df.columns:
        col_str = str(col).strip()

        if re.fullmatch(r"\d{4}", col_str):
            ano = int(col_str)

            if 2001 <= ano <= 2024:
                colunas_anos.append(col)

    if len(colunas_anos) == 0:
        raise ValueError("Não encontrei colunas de anos entre 2001 e 2024 no arquivo de casos.")

    return colunas_anos


def separar_codigo_nome_municipio(valor):
    """
    Recebe:
    '350010 ADAMANTINA'

    Retorna:
    cod_mun6 = '350010'
    municipio = 'ADAMANTINA'
    """

    if pd.isna(valor):
        return pd.Series([np.nan, np.nan])

    valor = str(valor).strip()

    m = re.match(r"^\s*(\d{6,7})\s+(.+?)\s*$", valor)

    if m:
        cod = m.group(1)[:6].zfill(6)
        nome = m.group(2).strip().upper()
        return pd.Series([cod, nome])

    cod = limpar_codigo_municipio(valor)

    nome = re.sub(r"^\s*\d{6,7}\s*", "", valor).strip().upper()

    if nome == "":
        nome = np.nan

    return pd.Series([cod, nome])


def padronizar_casos_wide(df, estado_forcado):
    """
    Transforma arquivo de casos do formato largo:

    Município | 2001 | 2002 | ... | 2024 | Total

    para formato longo:

    estado | cod_mun6 | municipio | ano | casos
    """

    df = df.copy()

    col_mun = detectar_coluna_municipio_casos_wide(df)
    colunas_anos = detectar_colunas_anos_casos_wide(df)

    print(f"\nPadronizando casos de {estado_forcado}")
    print(f"Coluna município detectada: {col_mun}")
    print(f"Anos detectados: {min(map(int, colunas_anos))}–{max(map(int, colunas_anos))}")
    print(f"Nº de colunas de anos usadas: {len(colunas_anos)}")

    df[["cod_mun6", "municipio"]] = df[col_mun].apply(separar_codigo_nome_municipio)

    df["estado"] = estado_forcado

    # Remove linhas sem código municipal válido, como Total, rodapé etc.
    df = df[df["cod_mun6"].notna()].copy()

    # Remove códigos que não são do estado forçado, se houver sujeira
    if estado_forcado == "SP":
        df = df[df["cod_mun6"].astype(str).str.startswith("35")].copy()
    elif estado_forcado == "RJ":
        df = df[df["cod_mun6"].astype(str).str.startswith("33")].copy()

    for col in colunas_anos:
        df[col] = df[col].apply(limpar_casos)

    casos_long = df.melt(
        id_vars=["estado", "cod_mun6", "municipio"],
        value_vars=colunas_anos,
        var_name="ano",
        value_name="casos"
    )

    casos_long["ano"] = casos_long["ano"].astype(int)
    casos_long["casos"] = pd.to_numeric(casos_long["casos"], errors="coerce").fillna(0)

    casos_long = (
        casos_long
        .groupby(["estado", "cod_mun6", "municipio", "ano"], as_index=False)["casos"]
        .sum()
    )

    return casos_long


def carregar_casos():
    sp_raw = ler_csv_auto(ARQ_CASOS_SP)
    rj_raw = ler_csv_auto(ARQ_CASOS_RJ)

    casos_sp = padronizar_casos_wide(sp_raw, estado_forcado="SP")
    casos_rj = padronizar_casos_wide(rj_raw, estado_forcado="RJ")

    casos = pd.concat([casos_sp, casos_rj], ignore_index=True)

    casos = casos[
        casos["ano"].between(2001, 2024)
    ].copy()

    print("\nResumo dos casos carregados:")
    resumo = (
        casos
        .groupby("estado")
        .agg(
            municipios_com_linha=("cod_mun6", "nunique"),
            anos=("ano", "nunique"),
            total_casos=("casos", "sum")
        )
        .reset_index()
    )
    print(resumo)

    return casos


# ============================================================
# GEOJSON
# ============================================================

def carregar_geojson_codigos(path, estado):
    path = Path(path)

    if not path.exists():
        print(f"GeoJSON não encontrado: {path}")
        return pd.DataFrame(columns=["estado", "cod_mun6", "municipio_geojson"])

    with open(path, "r", encoding="utf-8") as f:
        geo = json.load(f)

    linhas = []

    for feat in geo.get("features", []):
        props = feat.get("properties", {})

        cod = props.get("id", np.nan)
        nome = props.get("name", np.nan)

        linhas.append({
            "estado": estado,
            "cod_mun6": limpar_codigo_municipio(cod),
            "municipio_geojson": str(nome).strip().upper(),
        })

    return pd.DataFrame(linhas)


def carregar_geojsons():
    geo_rj = carregar_geojson_codigos(ARQ_GEOJSON_RJ, "RJ")
    geo_sp = carregar_geojson_codigos(ARQ_GEOJSON_SP, "SP")

    geojson = pd.concat([geo_rj, geo_sp], ignore_index=True)

    print("\nResumo GeoJSON:")
    print(
        geojson
        .groupby("estado")
        .agg(municipios_geojson=("cod_mun6", "nunique"))
        .reset_index()
    )

    return geojson


# ============================================================
# AUDITORIA POPULACIONAL
# ============================================================

def auditoria_estrutura(pop_wide):
    linhas = []

    for estado, df_estado in pop_wide.groupby("estado"):
        n_esperado = MUNICIPIOS_ESPERADOS.get(estado, np.nan)
        n_unicos = df_estado["cod_mun6"].nunique()

        linhas.append({
            "estado": estado,
            "n_linhas": len(df_estado),
            "n_municipios_unicos": n_unicos,
            "n_municipios_esperados": n_esperado,
            "diferenca_para_esperado": n_unicos - n_esperado if pd.notna(n_esperado) else np.nan,
            "duplicatas_cod_mun": df_estado.duplicated(["estado", "cod_mun6"]).sum(),
            "celulas_populacao_total": len(df_estado) * len(ANOS_POP),
            "celulas_populacao_ausentes": df_estado[ANOS_POP].isna().sum().sum(),
            "celulas_populacao_zero_ou_negativa": (df_estado[ANOS_POP] <= 0).sum().sum(),
            "menor_populacao": df_estado[ANOS_POP].min().min(),
            "maior_populacao": df_estado[ANOS_POP].max().max(),
        })

    return pd.DataFrame(linhas)


def auditoria_ausentes(pop_long):
    return (
        pop_long[pop_long["populacao"].isna()]
        .sort_values(["estado", "cod_mun6", "ano"])
        .copy()
    )


def auditoria_nao_positivos(pop_long):
    return (
        pop_long[
            pop_long["populacao"].notna() &
            (pop_long["populacao"] <= 0)
        ]
        .sort_values(["estado", "cod_mun6", "ano"])
        .copy()
    )


def auditoria_duplicados(pop_wide):
    return (
        pop_wide[
            pop_wide.duplicated(["estado", "cod_mun6"], keep=False)
        ]
        .sort_values(["estado", "cod_mun6"])
        .copy()
    )


def auditoria_saltos(pop_long, limiar):
    out = pop_long[
        pop_long["var_pop_pct"].notna() &
        (pop_long["var_pop_pct"].abs() >= limiar)
    ].copy()

    out["tipo_alerta"] = np.where(
        out["var_pop_pct"] > 0,
        f"Aumento anual >= {limiar}%",
        f"Queda anual >= {limiar}%"
    )

    return out.sort_values(["estado", "cod_mun6", "ano"])


def calcular_cagr(p_ini, p_fim, ano_ini, ano_fim):
    if pd.isna(p_ini) or pd.isna(p_fim):
        return np.nan

    if p_ini <= 0 or p_fim <= 0:
        return np.nan

    return (p_fim / p_ini) ** (1 / (ano_fim - ano_ini)) - 1


def auditoria_cagr(pop_wide):
    linhas = []

    for _, row in pop_wide.iterrows():
        for ano_ini, ano_fim in INTERVALOS_CAGR:
            p_ini = row[ano_ini]
            p_fim = row[ano_fim]

            cagr = calcular_cagr(p_ini, p_fim, ano_ini, ano_fim)

            if pd.isna(cagr):
                status = "Não calculado"
            elif abs(cagr * 100) >= LIMIAR_CAGR_EXTREMO:
                status = "CAGR extremo"
            else:
                status = "OK"

            linhas.append({
                "estado": row["estado"],
                "cod_mun6": row["cod_mun6"],
                "municipio": row["municipio"],
                "intervalo": f"{ano_ini}-{ano_fim}",
                "ano_ini": ano_ini,
                "ano_fim": ano_fim,
                "pop_ini": p_ini,
                "pop_fim": p_fim,
                "cagr": cagr,
                "cagr_pct": cagr * 100 if pd.notna(cagr) else np.nan,
                "status": status,
            })

    return pd.DataFrame(linhas)


def auditoria_geojson(pop_wide, geojson):
    pop_codes = pop_wide[["estado", "cod_mun6", "municipio"]].drop_duplicates()
    geo_codes = geojson[["estado", "cod_mun6", "municipio_geojson"]].drop_duplicates()

    pop_key = set(zip(pop_codes["estado"], pop_codes["cod_mun6"]))
    geo_key = set(zip(geo_codes["estado"], geo_codes["cod_mun6"]))

    linhas = []

    for estado, cod in sorted(geo_key - pop_key):
        municipio_geo = geo_codes.loc[
            (geo_codes["estado"] == estado) &
            (geo_codes["cod_mun6"] == cod),
            "municipio_geojson"
        ]

        linhas.append({
            "estado": estado,
            "cod_mun6": cod,
            "tipo_alerta": "Código no GeoJSON e ausente na população",
            "municipio_pop": np.nan,
            "municipio_geojson": municipio_geo.iloc[0] if len(municipio_geo) > 0 else np.nan,
        })

    for estado, cod in sorted(pop_key - geo_key):
        municipio_pop = pop_codes.loc[
            (pop_codes["estado"] == estado) &
            (pop_codes["cod_mun6"] == cod),
            "municipio"
        ]

        linhas.append({
            "estado": estado,
            "cod_mun6": cod,
            "tipo_alerta": "Código na população e ausente no GeoJSON",
            "municipio_pop": municipio_pop.iloc[0] if len(municipio_pop) > 0 else np.nan,
            "municipio_geojson": np.nan,
        })

    return pd.DataFrame(linhas)


# ============================================================
# PAINEL MUNICÍPIO-ANO: POPULAÇÃO + CASOS
# ============================================================

def montar_painel(pop_long, casos):
    painel = pop_long[
        pop_long["ano"].between(2001, 2024)
    ][["estado", "cod_mun6", "municipio", "ano", "populacao"]].copy()

    casos_aux = (
        casos
        .groupby(["estado", "cod_mun6", "ano"])["casos"]
        .sum()
        .reset_index()
    )

    painel = painel.merge(
        casos_aux,
        on=["estado", "cod_mun6", "ano"],
        how="left"
    )

    painel["casos"] = painel["casos"].fillna(0)

    painel["incidencia_100mil"] = np.where(
        painel["populacao"] > 0,
        100_000 * painel["casos"] / painel["populacao"],
        np.nan
    )

    return painel


def compatibilidade_pop_casos(pop_wide, casos):
    pop_codes = (
        pop_wide[["estado", "cod_mun6", "municipio"]]
        .drop_duplicates()
        .copy()
    )

    casos_codes = (
        casos[["estado", "cod_mun6"]]
        .drop_duplicates()
        .copy()
    )

    pop_key = set(zip(pop_codes["estado"], pop_codes["cod_mun6"]))
    casos_key = set(zip(casos_codes["estado"], casos_codes["cod_mun6"]))

    linhas = []

    for estado, cod in sorted(pop_key - casos_key):
        municipio = pop_codes.loc[
            (pop_codes["estado"] == estado) &
            (pop_codes["cod_mun6"] == cod),
            "municipio"
        ].iloc[0]

        linhas.append({
            "estado": estado,
            "cod_mun6": cod,
            "municipio": municipio,
            "tipo_alerta": "Tem população, mas não aparece no arquivo de casos; tratar como zero se confirmado no DATASUS"
        })

    for estado, cod in sorted(casos_key - pop_key):
        linhas.append({
            "estado": estado,
            "cod_mun6": cod,
            "municipio": np.nan,
            "tipo_alerta": "Tem casos, mas não aparece na população"
        })

    return pd.DataFrame(linhas)


# ============================================================
# LEI DOS PEQUENOS NÚMEROS
# ============================================================

def classificar_pequenos_numeros(pop):
    if pd.isna(pop):
        return "Sem população"

    if pop < POP_MUITO_PEQUENA:
        return "Muito sujeito à lei dos pequenos números (<10 mil)"

    if pop < POP_PEQUENA:
        return "Sujeito à lei dos pequenos números (10–20 mil)"

    if pop < POP_MODERADA:
        return "Atenção moderada (20–50 mil)"

    return "Menos sujeito (>=50 mil)"


def auditoria_pequenos_numeros_municipios(pop_wide, painel):
    pop2024 = (
        pop_wide[["estado", "cod_mun6", "municipio", 2024]]
        .rename(columns={2024: "pop_2024"})
        .copy()
    )

    casos_resumo = (
        painel
        .groupby(["estado", "cod_mun6"])
        .agg(
            casos_total_2001_2024=("casos", "sum"),
            media_casos_ano=("casos", "mean"),
            mediana_casos_ano=("casos", "median"),
            max_casos_ano=("casos", "max"),
            anos_com_zero_casos=("casos", lambda x: int((x == 0).sum())),
            anos_com_1a5_casos=("casos", lambda x: int(((x >= 1) & (x <= 5)).sum())),
            max_incidencia_100mil=("incidencia_100mil", "max"),
            media_incidencia_100mil=("incidencia_100mil", "mean"),
        )
        .reset_index()
    )

    out = pop2024.merge(
        casos_resumo,
        on=["estado", "cod_mun6"],
        how="left"
    )

    out["casos_total_2001_2024"] = out["casos_total_2001_2024"].fillna(0)
    out["media_casos_ano"] = out["media_casos_ano"].fillna(0)
    out["mediana_casos_ano"] = out["mediana_casos_ano"].fillna(0)
    out["max_casos_ano"] = out["max_casos_ano"].fillna(0)
    out["anos_com_zero_casos"] = out["anos_com_zero_casos"].fillna(24).astype(int)
    out["anos_com_1a5_casos"] = out["anos_com_1a5_casos"].fillna(0).astype(int)

    out["classe_pequenos_numeros"] = out["pop_2024"].apply(classificar_pequenos_numeros)

    out["flag_pop_menor_10k"] = out["pop_2024"] < POP_MUITO_PEQUENA
    out["flag_pop_menor_20k"] = out["pop_2024"] < POP_PEQUENA
    out["flag_pop_menor_50k"] = out["pop_2024"] < POP_MODERADA

    out["flag_media_casos_baixa"] = out["media_casos_ano"] <= 5
    out["flag_muitos_zeros"] = out["anos_com_zero_casos"] >= 12

    out["prioridade_revisao"] = np.select(
        [
            (out["pop_2024"] < POP_MUITO_PEQUENA) &
            (out["max_incidencia_100mil"] >= LIMIAR_INCIDENCIA_MUITO_ALTA),

            (out["pop_2024"] < POP_PEQUENA) &
            (out["max_incidencia_100mil"] >= LIMIAR_INCIDENCIA_ALTA),

            (out["pop_2024"] < POP_MODERADA) &
            (out["media_casos_ano"] <= 5),
        ],
        [
            "Alta: população muito pequena e incidência máxima muito alta",
            "Alta: pequeno denominador e incidência alta",
            "Média: população moderada/pequena e poucos casos médios",
        ],
        default="Baixa"
    )

    out = out.sort_values(
        ["estado", "pop_2024", "max_incidencia_100mil"],
        ascending=[True, True, False]
    )

    return out


def auditoria_pequenos_numeros_ano(painel):
    out = painel.copy()

    out["classe_pequenos_numeros_ano"] = out["populacao"].apply(classificar_pequenos_numeros)

    out["flag_pequeno_denominador"] = out["populacao"] < POP_PEQUENA
    out["flag_incidencia_alta"] = out["incidencia_100mil"] >= LIMIAR_INCIDENCIA_ALTA
    out["flag_incidencia_muito_alta"] = out["incidencia_100mil"] >= LIMIAR_INCIDENCIA_MUITO_ALTA

    out["flag_poucos_casos_taxa_instavel"] = (
        (out["populacao"] < POP_PEQUENA) &
        (out["casos"].between(1, 5)) &
        (out["incidencia_100mil"] >= LIMIAR_INCIDENCIA_ALTA)
    )

    out["tipo_alerta"] = np.select(
        [
            out["flag_poucos_casos_taxa_instavel"],
            out["flag_pequeno_denominador"] & out["flag_incidencia_muito_alta"],
            out["flag_pequeno_denominador"] & out["flag_incidencia_alta"],
            out["flag_pequeno_denominador"],
        ],
        [
            "Incidência alta com poucos casos em pequeno denominador",
            "Incidência muito alta em pequeno denominador",
            "Incidência alta em pequeno denominador",
            "Pequeno denominador populacional",
        ],
        default="Sem alerta"
    )

    out_alertas = out[out["tipo_alerta"] != "Sem alerta"].copy()

    out_alertas = out_alertas.sort_values(
        ["estado", "cod_mun6", "ano"]
    )

    return out_alertas


# ============================================================
# RESUMOS
# ============================================================

def montar_resumo_pequenos_numeros(pequenos_municipios):
    resumo = (
        pequenos_municipios
        .groupby(["estado", "classe_pequenos_numeros"])
        .agg(
            n_municipios=("cod_mun6", "count"),
            pop_min=("pop_2024", "min"),
            pop_max=("pop_2024", "max"),
            media_casos_ano_media=("media_casos_ano", "mean"),
            max_incidencia_100mil_max=("max_incidencia_100mil", "max"),
        )
        .reset_index()
    )

    total_estado = (
        pequenos_municipios
        .groupby("estado")["cod_mun6"]
        .count()
        .reset_index(name="total_estado")
    )

    resumo = resumo.merge(total_estado, on="estado", how="left")
    resumo["perc_municipios_estado"] = 100 * resumo["n_municipios"] / resumo["total_estado"]

    return resumo.sort_values(["estado", "classe_pequenos_numeros"])


def montar_resumo_alertas(
    ausentes,
    nao_positivos,
    duplicados,
    saltos10,
    saltos20,
    cagr,
    geojson_alertas,
    compat_pop_casos,
    pequenos_municipios,
    pequenos_ano
):
    linhas = [
        {
            "tipo_alerta": "População ausente",
            "n_registros": len(ausentes),
        },
        {
            "tipo_alerta": "População zero ou negativa",
            "n_registros": len(nao_positivos),
        },
        {
            "tipo_alerta": "Município duplicado na população",
            "n_registros": len(duplicados),
        },
        {
            "tipo_alerta": f"Salto populacional anual >= {LIMIAR_SALTO_ANUAL_10}%",
            "n_registros": len(saltos10),
        },
        {
            "tipo_alerta": f"Salto populacional anual >= {LIMIAR_SALTO_ANUAL_20}%",
            "n_registros": len(saltos20),
        },
        {
            "tipo_alerta": f"CAGR absoluto >= {LIMIAR_CAGR_EXTREMO}% ao ano",
            "n_registros": len(cagr[cagr["status"] == "CAGR extremo"]),
        },
        {
            "tipo_alerta": "Inconsistência população × GeoJSON",
            "n_registros": len(geojson_alertas),
        },
        {
            "tipo_alerta": "Inconsistência população × casos",
            "n_registros": len(compat_pop_casos),
        },
        {
            "tipo_alerta": "Municípios com população 2024 < 10 mil",
            "n_registros": int((pequenos_municipios["pop_2024"] < POP_MUITO_PEQUENA).sum()),
        },
        {
            "tipo_alerta": "Municípios com população 2024 < 20 mil",
            "n_registros": int((pequenos_municipios["pop_2024"] < POP_PEQUENA).sum()),
        },
        {
            "tipo_alerta": "Municípios-ano com pequeno denominador e incidência alta",
            "n_registros": int(
                pequenos_ano["tipo_alerta"].isin([
                    "Incidência alta com poucos casos em pequeno denominador",
                    "Incidência muito alta em pequeno denominador",
                    "Incidência alta em pequeno denominador",
                ]).sum()
            ),
        },
    ]

    return pd.DataFrame(linhas)


# ============================================================
# EXECUÇÃO
# ============================================================

print("============================================================")
print("CARREGANDO POPULAÇÃO")
print("============================================================")

pop_wide, abas_usadas, abas_ignoradas = carregar_populacao(ARQ_POP)
pop_long = populacao_wide_para_long(pop_wide)

print("\nResumo da população carregada:")
print(
    pop_wide
    .groupby("estado")
    .agg(
        municipios=("cod_mun6", "nunique"),
        linhas=("cod_mun6", "count")
    )
    .reset_index()
)

print("\n============================================================")
print("CARREGANDO CASOS")
print("============================================================")

casos = carregar_casos()

print("\nPrimeiras linhas dos casos padronizados:")
mostrar(casos, n=10)

print("\nMunicípios por estado nos casos:")
print(casos.groupby("estado")["cod_mun6"].nunique())

print("\nTotal de casos por estado:")
print(casos.groupby("estado")["casos"].sum())

print("\n============================================================")
print("CARREGANDO GEOJSON")
print("============================================================")

geojson = carregar_geojsons()

print("\n============================================================")
print("MONTANDO PAINEL POPULAÇÃO + CASOS")
print("============================================================")

painel = montar_painel(pop_long, casos)

print("\nResumo do painel:")
print(
    painel
    .groupby("estado")
    .agg(
        municipios=("cod_mun6", "nunique"),
        anos=("ano", "nunique"),
        casos_total=("casos", "sum"),
        incidencia_media=("incidencia_100mil", "mean")
    )
    .reset_index()
)

print("\n============================================================")
print("EXECUTANDO AUDITORIAS")
print("============================================================")

estrutura = auditoria_estrutura(pop_wide)
ausentes = auditoria_ausentes(pop_long)
nao_positivos = auditoria_nao_positivos(pop_long)
duplicados = auditoria_duplicados(pop_wide)

saltos10 = auditoria_saltos(pop_long, LIMIAR_SALTO_ANUAL_10)
saltos20 = auditoria_saltos(pop_long, LIMIAR_SALTO_ANUAL_20)

cagr = auditoria_cagr(pop_wide)
cagr_extremo = cagr[cagr["status"] == "CAGR extremo"].copy()

geojson_alertas = auditoria_geojson(pop_wide, geojson)
compat_pop_casos = compatibilidade_pop_casos(pop_wide, casos)

pequenos_municipios = auditoria_pequenos_numeros_municipios(pop_wide, painel)
pequenos_ano = auditoria_pequenos_numeros_ano(painel)

resumo_pequenos = montar_resumo_pequenos_numeros(pequenos_municipios)

resumo_alertas = montar_resumo_alertas(
    ausentes=ausentes,
    nao_positivos=nao_positivos,
    duplicados=duplicados,
    saltos10=saltos10,
    saltos20=saltos20,
    cagr=cagr,
    geojson_alertas=geojson_alertas,
    compat_pop_casos=compat_pop_casos,
    pequenos_municipios=pequenos_municipios,
    pequenos_ano=pequenos_ano,
)

# Possíveis problemas populacionais consolidados
possiveis_problemas_pop = pd.concat([
    saltos10.assign(origem_alerta="Salto anual >= 10%"),
    cagr_extremo.rename(columns={
        "pop_ini": "populacao_anterior",
        "pop_fim": "populacao"
    }).assign(origem_alerta="CAGR extremo"),
], ignore_index=True, sort=False)

municipios_pequenos_prioritarios = pequenos_municipios[
    pequenos_municipios["prioridade_revisao"].isin([
        "Alta: população muito pequena e incidência máxima muito alta",
        "Alta: pequeno denominador e incidência alta",
        "Média: população moderada/pequena e poucos casos médios",
    ])
].copy()

municipios_pequenos_prioritarios = municipios_pequenos_prioritarios.sort_values(
    ["estado", "prioridade_revisao", "pop_2024"],
    ascending=[True, True, True]
)


# ============================================================
# IMPRESSÃO DOS RESULTADOS
# ============================================================

print("\n==============================")
print("RESUMO DA ESTRUTURA")
print("==============================")
mostrar(estrutura, n=50)

print("\n==============================")
print("RESUMO DOS ALERTAS")
print("==============================")
mostrar(resumo_alertas, n=50)

print("\n==============================")
print("RESUMO DA LEI DOS PEQUENOS NÚMEROS")
print("==============================")
mostrar(resumo_pequenos, n=50)

print("\n==============================")
print("COMPATIBILIDADE POPULAÇÃO × CASOS")
print("==============================")
mostrar(compat_pop_casos, n=50)

print("\n==============================")
print("POSSÍVEIS PROBLEMAS POPULACIONAIS")
print("==============================")
mostrar(possiveis_problemas_pop, n=50)

print("\n==============================")
print("MUNICÍPIOS MAIS SUJEITOS À LEI DOS PEQUENOS NÚMEROS")
print("==============================")
mostrar(pequenos_municipios.sort_values(["estado", "pop_2024"]), n=50)

print("\n==============================")
print("MUNICÍPIOS-ANO COM INCIDÊNCIA INSTÁVEL")
print("==============================")
mostrar(pequenos_ano, n=50)


# ============================================================
# EXPORTAÇÃO PARA EXCEL
# ============================================================

print("\n============================================================")
print("EXPORTANDO RESULTADOS")
print("============================================================")

with pd.ExcelWriter(OUTFILE, engine="openpyxl") as writer:
    pd.DataFrame({
        "abas_usadas": pd.Series(abas_usadas),
        "abas_ignoradas": pd.Series(abas_ignoradas),
    }).to_excel(writer, sheet_name=safe_sheet_name("Abas"), index=False)

    estrutura.to_excel(writer, sheet_name=safe_sheet_name("Resumo_estrutura"), index=False)
    resumo_alertas.to_excel(writer, sheet_name=safe_sheet_name("Resumo_alertas"), index=False)
    resumo_pequenos.to_excel(writer, sheet_name=safe_sheet_name("Resumo_pequenos_num"), index=False)

    pop_wide.to_excel(writer, sheet_name=safe_sheet_name("Pop_wide_padronizada"), index=False)
    pop_long.to_excel(writer, sheet_name=safe_sheet_name("Pop_long"), index=False)
    casos.to_excel(writer, sheet_name=safe_sheet_name("Casos_long"), index=False)
    painel.to_excel(writer, sheet_name=safe_sheet_name("Painel_pop_casos"), index=False)

    ausentes.to_excel(writer, sheet_name=safe_sheet_name("Pop_ausente"), index=False)
    nao_positivos.to_excel(writer, sheet_name=safe_sheet_name("Pop_nao_positiva"), index=False)
    duplicados.to_excel(writer, sheet_name=safe_sheet_name("Duplicados"), index=False)

    saltos10.to_excel(writer, sheet_name=safe_sheet_name("Saltos_10pct"), index=False)
    saltos20.to_excel(writer, sheet_name=safe_sheet_name("Saltos_20pct"), index=False)

    cagr.to_excel(writer, sheet_name=safe_sheet_name("CAGR_intervalos"), index=False)
    cagr_extremo.to_excel(writer, sheet_name=safe_sheet_name("CAGR_extremo"), index=False)

    geojson_alertas.to_excel(writer, sheet_name=safe_sheet_name("GeoJSON_alertas"), index=False)
    compat_pop_casos.to_excel(writer, sheet_name=safe_sheet_name("Compat_pop_casos"), index=False)

    pequenos_municipios.to_excel(writer, sheet_name=safe_sheet_name("Pequenos_municipios"), index=False)
    municipios_pequenos_prioritarios.to_excel(writer, sheet_name=safe_sheet_name("Pequenos_prioritarios"), index=False)
    pequenos_ano.to_excel(writer, sheet_name=safe_sheet_name("Pequenos_mun_ano"), index=False)

    possiveis_problemas_pop.to_excel(writer, sheet_name=safe_sheet_name("Problemas_pop"), index=False)

print(f"\nArquivo salvo em: {OUTFILE}")

CARREGANDO POPULAÇÃO
Arquivo de população usado: PopulaçãoCorrigida_interpolada_CAGR_FINAL.xlsx

Avaliando aba: População RJ
Aba usada: População RJ
  Coluna código detectada: Cod_mun
  Coluna município detectada: Municipio

Avaliando aba: População SP
Aba usada: População SP
  Coluna código detectada: Cod_mun
  Coluna município detectada: Municipio

Resumo da população carregada:
  estado  municipios  linhas
0     RJ          92      92
1     SP         645     645

CARREGANDO CASOS

Arquivo lido: Casos-TB-SP-NOVO.csv
Separador: ; | Encoding: latin1
Nº linhas: 646 | Nº colunas: 27
Primeiras colunas: ['Município de notificação', '2001', '2002', '2003', '2004', '2005', '2006', '2007']

Arquivo lido: Casos-TB-RJ-NOVO.csv
Separador: ; | Encoding: latin1
Nº linhas: 94 | Nº colunas: 28
Primeiras colunas: ['Município de notificação', '2001', '2002', '2003', '2004', '2005', '2006', '2007']

Padronizando casos de SP
Coluna município detectada: Município de notificação
Anos detectados: 2001–202

,estado,cod_mun6,municipio,ano,casos
0,SP,350010,ADAMANTINA,2001,4.0
1,SP,350010,ADAMANTINA,2002,9.0
2,SP,350010,ADAMANTINA,2003,4.0
3,SP,350010,ADAMANTINA,2004,6.0
4,SP,350010,ADAMANTINA,2005,2.0
5,SP,350010,ADAMANTINA,2006,5.0
6,SP,350010,ADAMANTINA,2007,5.0
7,SP,350010,ADAMANTINA,2008,6.0
8,SP,350010,ADAMANTINA,2009,5.0
9,SP,350010,ADAMANTINA,2010,5.0



Municípios por estado nos casos:
estado
RJ     92
SP    644
Name: cod_mun6, dtype: int64

Total de casos por estado:
estado
RJ    356603.0
SP    479036.0
Name: casos, dtype: float64

CARREGANDO GEOJSON
GeoJSON não encontrado: RJ_geojs-33-mun.json
GeoJSON não encontrado: SP_geojs-35-mun.json

Resumo GeoJSON:
Empty DataFrame
Columns: [estado, municipios_geojson]
Index: []

MONTANDO PAINEL POPULAÇÃO + CASOS

Resumo do painel:
  estado  municipios  anos  casos_total  incidencia_media
0     RJ          92    24     356603.0         48.697294
1     SP         645    24     479036.0         33.390964

EXECUTANDO AUDITORIAS

RESUMO DA ESTRUTURA


,estado,n_linhas,n_municipios_unicos,n_municipios_esperados,diferenca_para_esperado,duplicatas_cod_mun,celulas_populacao_total,celulas_populacao_ausentes,celulas_populacao_zero_ou_negativa,menor_populacao,maior_populacao
0,RJ,92,92,92,0,0,2300,0,0,4886.0,6730729.0
1,SP,645,645,645,0,0,16125,0,0,795.0,11895578.0



RESUMO DOS ALERTAS


...


MUNICÍPIOS MAIS SUJEITOS À LEI DOS PEQUENOS NÚMEROS


,estado,cod_mun6,municipio,pop_2024,casos_total_2001_2024,media_casos_ano,mediana_casos_ano,max_casos_ano,anos_com_zero_casos,anos_com_1a5_casos,max_incidencia_100mil,media_incidencia_100mil,classe_pequenos_numeros,flag_pop_menor_10k,flag_pop_menor_20k,flag_pop_menor_50k,flag_media_casos_baixa,flag_muitos_zeros,prioridade_revisao
37,RJ,330245,MACUCO,5602.0,51.0,2.125000,2.0,7.0,6,16,124.955373,40.123692,Muito sujeito à lei dos pequenos números (<10 ...,True,True,True,True,False,Alta: pequeno denominador e incidência alta
75,RJ,330513,SAO JOSE DE UBA,7318.0,35.0,1.458333,1.5,5.0,6,18,70.721358,21.139465,Muito sujeito à lei dos pequenos números (<10 ...,True,True,True,True,False,Média: população moderada/pequena e poucos cas...
35,RJ,330230,LAJE DO MURIAE,7584.0,95.0,3.958333,4.0,8.0,2,17,108.680886,52.652557,Muito sujeito à lei dos pequenos números (<10 ...,True,True,True,True,False,Alta: pequeno denominador e incidência alta
78,RJ,330530,SAO SEBASTIAO DO ALTO,7993.0,74.0,3.083333,2.5,10.0,2,20,117.674747,36.538955,Muito sujeito à lei dos pequenos números (<10 ...,True,True,True,True,False,Alta: pequeno denominador e incidência alta
20,RJ,330130,CASIMIRO DE ABREU,9048.0,273.0,11.375000,9.0,32.0,0,3,401.253918,136.432492,Muito sujeito à lei dos pequenos números (<10 ...,True,True,True,False,False,Alta: população muito pequena e incidência máx...
65,RJ,330450,RIO DAS FLORES,9267.0,32.0,1.333333,1.0,4.0,7,17,46.723514,15.730901,Muito sujeito à lei dos pequenos números (<10 ...,True,True,True,True,False,Média: população moderada/pequena e poucos cas...
89,RJ,330615,VARRE-SAI,10563.0,8.0,0.333333,0.0,2.0,18,6,20.848535,3.515536,Sujeito à lei dos pequenos números (10–20 mil),False,True,True,True,True,Média: população moderada/pequena e poucos cas...
68,RJ,330460,SANTA MARIA MADALENA,10580.0,35.0,1.458333,1.5,5.0,10,14,47.943235,14.107100,Sujeito à lei dos pequenos números (10–20 mil),False,True,True,True,False,Média: população moderada/pequena e poucos cas...
86,RJ,330590,TRAJANO DE MORAES,10654.0,39.0,1.625000,1.0,10.0,7,15,99.373944,15.944631,Sujeito à lei dos pequenos números (10–20 mil),False,True,True,True,False,Média: população moderada/pequena e poucos cas...
23,RJ,330160,DUAS BARRAS,11355.0,85.0,3.541667,3.5,7.0,3,16,65.863756,32.726870,Sujeito à lei dos pequenos números (10–20 mil),False,True,True,True,False,Média: população moderada/pequena e poucos cas...



MUNICÍPIOS-ANO COM INCIDÊNCIA INSTÁVEL


,estado,cod_mun6,municipio,ano,populacao,casos,incidencia_100mil,classe_pequenos_numeros_ano,flag_pequeno_denominador,flag_incidencia_alta,flag_incidencia_muito_alta,flag_poucos_casos_taxa_instavel,tipo_alerta
24,RJ,330015,APERIBE,2001,8214.0,2.0,24.348673,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
25,RJ,330015,APERIBE,2002,8416.0,5.0,59.410646,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
26,RJ,330015,APERIBE,2003,8622.0,2.0,23.196474,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
27,RJ,330015,APERIBE,2004,8833.0,4.0,45.284728,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
28,RJ,330015,APERIBE,2005,9049.0,6.0,66.305669,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
29,RJ,330015,APERIBE,2006,9271.0,1.0,10.786323,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
30,RJ,330015,APERIBE,2007,9498.0,3.0,31.585597,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
31,RJ,330015,APERIBE,2008,9731.0,1.0,10.276436,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
32,RJ,330015,APERIBE,2009,9969.0,6.0,60.186578,Muito sujeito à lei dos pequenos números (<10 ...,True,False,False,False,Pequeno denominador populacional
33,RJ,330015,APERIBE,2010,10213.0,3.0,29.374327,Sujeito à lei dos pequenos números (10–20 mil),True,False,False,False,Pequeno denominador populacional



EXPORTANDO RESULTADOS

Arquivo salvo em: Auditoria_populacao_e_pequenos_numeros_TB_SP_RJ_CORRIGIDA.xlsx
